In [ ]:


import warnings
warnings.filterwarnings('ignore')  # hide harmless warning messages so the output stays readable

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')  # makes charts look a bit nicer

print('Libraries loaded.')

Libraries loaded.


## Step 1: Load the data and look at it
We read the CSV file into a table called a "DataFrame". `df` is just a short name for it

In [4]:
df = pd.read_csv('../data/raw/telco.csv')

print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

df.head()   # shows the first 5 rows, so we can see what the data looks like

Rows: 7043
Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
# .info() tells us the column names, how many values each has, and what data TYPE each column is
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [7]:
# Let's look for rows where TotalCharges is not a normal number.
# .str.strip() removes extra spaces. If what's left is an empty string '', that's our problem row.

is_blank = df['TotalCharges'].str.strip() == ''
print('Number of blank TotalCharges values:', is_blank.sum())

# show those exact rows so we can see what they have in common
df.loc[is_blank, ['tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

Number of blank TotalCharges values: 11


,tenure,MonthlyCharges,TotalCharges,Churn
488,0,52.55,,No
753,0,20.25,,No
936,0,80.85,,No
1082,0,25.75,,No
1340,0,56.05,,No
3331,0,19.85,,No
3826,0,25.35,,No
4380,0,20.00,,No
5218,0,19.70,,No
6670,0,73.35,,No




All the 11 customers have a tenure of 0, meaning they are new customers who have not yet completed a month of service. This could explain why their TotalCharges values are blank, since they may not have accumulated any charges yet. However, they already have monthly charges, which suggests that they have subscribed to a service but have not yet accumulated a total billing amount.

## Step 2: Fix the data types
Now we clean up the data before we do anything else with it. Two small fixes:

1. Turn `TotalCharges` into real numbers. The blank values become `NaN` ("Not a Number"), which is
   pandas' way of marking a missing value. We do NOT fill them in yet - that happens later, and only
   using the training data, so we don't cheat.
2. Turn the target column `Churn` (currently "Yes"/"No") into 1/0, because machine learning models
   need numbers, not words, for the thing they are predicting.

In [8]:
# pd.to_numeric turns text into numbers. errors='coerce' means: if something can't be
# turned into a number, replace it with NaN instead of crashing.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('Missing values in TotalCharges now:', df['TotalCharges'].isna().sum())

# customerID is just a random ID string, like a serial number - it can't help predict anything,
# so we remove it.
df = df.drop(columns=['customerID'])

# Turn the target into 1 = churned, 0 = did not churn
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

df.head()

Missing values in TotalCharges now: 11


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


## Step 3: Split into training data and testing data


In [9]:
from sklearn.model_selection import train_test_split

# X = the columns we use to make a prediction (everything except Churn)
# y = the thing we are trying to predict (the Churn column)
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% of the data goes into the test set, 80% into training
    stratify=y,           # keep the same churn percentage in both sets
    random_state=42,      # a fixed number so we get the SAME split every time we run this cell
)

print('Training rows:', X_train.shape[0])
print('Testing rows: ', X_test.shape[0])
print()
print('Churn rate in full data:    ', round(y.mean(), 3))
print('Churn rate in training set:', round(y_train.mean(), 3))
print('Churn rate in test set:    ', round(y_test.mean(), 3))

Training rows: 5634
Testing rows:  1409

Churn rate in full data:     0.265
Churn rate in training set: 0.265
Churn rate in test set:     0.265


## Step 4: Build the preprocessing pipeline



In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# List the numeric columns and the category (text) columns separately,
# because they each need different treatment.

numeric_columns = ['tenure', 'MonthlyCharges', 'TotalCharges']

category_columns = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
]

# Steps for the numeric columns: first fill in missing values, then scale them
numeric_steps = Pipeline([
    ('fill_missing', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])

# Step for the category columns: turn each category into its own 0/1 column
category_steps = OneHotEncoder(handle_unknown='ignore')
# handle_unknown='ignore' means: if the test set has a category the training set never saw,
# don't crash - just treat it as all zeros.

# Now combine both: apply numeric_steps to the numeric columns,
# and category_steps to the category columns
preprocessing = ColumnTransformer([
    ('numbers', numeric_steps, numeric_columns),
    ('categories', category_steps, category_columns),
])

print('Pipeline built.')

Pipeline built.


In [12]:
# quick check: after preprocessing, is there any missing data left, and how many columns do we have now?
X_train_ready = preprocessing.transform(X_train)

print('Shape after preprocessing:', X_train_ready.shape)
print('Any missing values left:', np.isnan(X_train_ready).any())

Shape after preprocessing: (5634, 46)
Any missing values left: False
